# utm-zone-finder demo

Find the UTM zone and EPSG code of ten real cities. Data: `examples/cities.geojson` (WGS84, GeoJSON order x = longitude, y = latitude). The API takes `(lat, lon)`.

In [ ]:
import importlib.util
import json
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec('utm_zone_finder') is None:  # e.g. on Colab
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/AmmarYasser455/utm-zone-finder'], check=True)

for p in (Path('examples/cities.geojson'), Path('cities.geojson')):
    if p.exists():
        cities = json.loads(p.read_text(encoding='utf-8'))
        break
else:
    url = 'https://raw.githubusercontent.com/AmmarYasser455/utm-zone-finder/main/examples/cities.geojson'
    cities = json.loads(urllib.request.urlopen(url).read().decode('utf-8'))
len(cities['features'])

## Zone of each city

Bergen is in the Norway exception (32V), Svalbard uses only zones 31, 33, 35 and 37.

In [ ]:
from utm_zone_finder import zone_for_point

zones = {}
for f in cities['features']:
    lon, lat = f['geometry']['coordinates']
    z = zone_for_point(lat, lon)
    zones[f['properties']['name']] = z
    print(f"{f['properties']['name']:<13} {z.name:<4} EPSG:{z.epsg}")
assert zones['Bergen'].epsg == 32632
assert zones['Longyearbyen'].epsg == 32633

## Bounding box and whole file

A box can span several zones; `zone_for_bbox` returns the zone of its centre and `zones_spanned` lists all of them.

In [ ]:
from utm_zone_finder import bbox_from_geojson, zone_for_bbox, zones_spanned

bbox = bbox_from_geojson(cities)  # (min_lon, min_lat, max_lon, max_lat)
print(bbox)
print(zone_for_bbox(*bbox).name, [z.name for z in zones_spanned(*bbox)][:5], '...')
print([z.name for z in zones_spanned(-1, 50, 1, 52)])